In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
from trend_analisys_rsi_markov import TrendAnalyzer
import warnings
from scripts.assetsRoster import carteira_AC, carteira_HB, others, carteira_EXC
warnings.filterwarnings('ignore')


In [2]:
asset_ids = [
    'bitcoin', 'ethereum', 'binancecoin', 'ripple', 'cardano',
    'solana', 'polkadot'
]

In [3]:
data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/'
btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv'
ssr_data_path='/Users/valter.rebelo/MissionControl/data/onchainData/BTC_SSR.csv'

In [8]:
analyzer = TrendAnalyzer(
    asset_ids=['bitcoin', 'ethereum'],
    data_path=data_path,
    btc_data_path=btc_data_path,
    ssr_data_path=ssr_data_path,
    use_btc_adjusted=True,
    verbose=True,
    lookback_days=180,
    trend_metrics_lookback=365,
    markov_model_name='20250325_144631'
)

2025-04-01 15:35:37,139 - INFO - Loaded Markov volatility model: 20250325_144631
2025-04-01 15:35:37,143 - INFO - Successfully loaded SSR data with 2353 rows.


Data loaded successfully: 4107 records
Training set: 3285 records
Test set: 822 records
Model metadata:
  Saved on: 2025-03-25
  Data range: 2014-01-01 to 2025-03-24
  Records: 4101 total, 3075 train, 1026 test
Model loaded from /Users/valter.rebelo/MissionControl/models/markov_volatility_model_20250325_144631.pkl


In [9]:

# Analyze assets
analyzer.analyze_multiple_assets()


Computing lookback metrics: 100%|██████████| 2/2 [00:00<00:00, 28.01it/s]
2025-04-01 15:35:44,682 - INFO - Trend analysis completed for multiple assets.


,Ticker,Short Term Trend (USD),Medium Term Trend (USD),Long Term Trend (USD),Overall Trend (USD),Short Term Trend (BTC),Medium Term Trend (BTC),Long Term Trend (BTC),Overall Trend (BTC),Sharpe Ratio (USD),...,Current Trend Sortino (USD),Current Trend Mean Return (USD),Current Trend Median Return (USD),Current Trend Skew (USD),Current Trend Sharpe (BTC),Current Trend Sortino (BTC),Current Trend Mean Return (BTC),Current Trend Median Return (BTC),Current Trend Skew (BTC),Latest Date
0,BTC,Strong Bear,Strong Bear,Weak Bull,Weak Bear,None,None,None,None,1.102142,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-30
1,ETH,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,1.164017,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-30


In [ ]:
analyzer.asset_data.get('ethereum').get('trend_metrics')

In [17]:
start_date='2024-01-1'
end_date='2025-03-30'

# Create a BTC-only trend-following portfolio
analyzer.create_portfolio(
    portfolio_name='btc_trend_following',
    usd_conditions={'Short Term (USD)': ['Weak Bull','Strong Bull']},
    btc_only=True,
    btc_trend_gating=None,
    rsi_conditions_usd=True,
    use_ssr_signal=True,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='big_wins_i',
    btc_conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=False,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='big_wins_ii',
    btc_conditions={'Short Term (BTC)': ['Weak Bull','Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=True,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)


# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='fat_tails',
    btc_conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Weak Bull','Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=True,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='experiment_i',
    btc_conditions={'Short Term (BTC)': ['Weak Bull','Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bear','Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=False,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)


########################################################

# Backtest the altcoin portfolio
results_alt = analyzer.backtest_portfolio(
    portfolio_name='big_wins_i',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.005,
    signal_threshold=100
)

results_alt = analyzer.backtest_portfolio(
    portfolio_name='experiment_i',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.005,
    signal_threshold=80
)

# Backtest the BTC trend-following portfolio
results_btc = analyzer.backtest_portfolio(
    portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.001,
    signal_threshold=50
)

2025-04-01 15:42:10,878 - WARNING - Portfolio btc_trend_following already exists. Overwriting.
2025-04-01 15:42:10,882 - INFO - Portfolio 'btc_trend_following' created with criteria: BTC gating=None, USD conditions={'Short Term (USD)': ['Weak Bull', 'Strong Bull']}, BTC conditions=None, RSI USD=True, RSI BTC=False, volatility filter=True, volatility weight=1, SSR signal=True, SSR gate=False, BTC RSI gate=False, BTC RSI signal=False, Follow portfolio=None, BTC only=True
2025-04-01 15:42:10,883 - WARNING - Portfolio big_wins_i already exists. Overwriting.
2025-04-01 15:42:10,884 - INFO - Portfolio 'big_wins_i' created with criteria: BTC gating=None, USD conditions={'Short Term (USD)': ['Strong Bull']}, BTC conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Strong Bull']}, RSI USD=True, RSI BTC=True, volatility filter=True, volatility weight=1, SSR signal=False, SSR gate=False, BTC RSI gate=False, BTC RSI signal=False, Follow portfolio=btc_trend_following, BTC only=False


In [18]:


# Plot performance for a specific date range
fig = analyzer.plot_individual_asset_performance(
    portfolio_name='big_wins_i',
    btc_trend_portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    show_plot=True,
   #asset_filter=['SOL']
)


2025-04-01 15:42:12,499 - INFO - Recalculating backtest for portfolio 'big_wins_i' from 2024-01-1 to 2025-03-30.
2025-04-01 15:42:12,510 - INFO - Using signals from BTC-only portfolio 'btc_trend_following'
2025-04-01 15:42:12,864 - INFO - Recalculating backtest for portfolio 'btc_trend_following' from 2024-01-1 to 2025-03-30.
2025-04-01 15:42:12,872 - INFO - Entering BTC position on 2024-01-08 00:00:00: Cost = 10.00
2025-04-01 15:42:12,874 - INFO - Exiting BTC position on 2024-01-09 00:00:00: Cost = 9.99
2025-04-01 15:42:12,875 - INFO - Entering BTC position on 2024-01-10 00:00:00: Cost = 9.98
2025-04-01 15:42:12,879 - INFO - Exiting BTC position on 2024-01-13 00:00:00: Cost = 9.29
2025-04-01 15:42:12,880 - INFO - Entering BTC position on 2024-01-14 00:00:00: Cost = 9.28
2025-04-01 15:42:12,883 - INFO - Exiting BTC position on 2024-01-16 00:00:00: Cost = 9.24
2025-04-01 15:42:12,893 - INFO - Entering BTC position on 2024-01-30 00:00:00: Cost = 9.23
2025-04-01 15:42:12,914 - INFO - Exit


Performance Metrics:
BTC Buy & Hold: Return = 86.46%, Max Drawdown = -26.23%
BTC Trend (btc_trend_following): Return = 87.26%, Max Drawdown = -19.76%

Individual Asset Performance:

|     | Total Return   | Max Drawdown   |   Sharpe Ratio |   Sortino Ratio |   Number of Trades | Win Rate   | Avg Win   | Avg Loss   |   Avg Holding Days | Total Costs   | First Trade         | Last Trade          |   Trading Days |
|:----|:---------------|:---------------|---------------:|----------------:|-------------------:|:-----------|:----------|:-----------|-------------------:|:--------------|:--------------------|:--------------------|---------------:|
| ETH | -0.53%         | -23.29%        |            5.5 |           13.48 |                 14 | 35.71%     | 3.30%     | -1.81%     |                3.8 | $1,435.62     | 2024-01-13 00:00:00 | 2024-12-16 00:00:00 |             53 |

Detailed Trade History:

ETH Trades:
|   trade_id | entry_date          | exit_date           |   holding_days |  

In [12]:
results_btc.get('metrics')

{'total_return': 0.872596158079114,
 'max_drawdown': -0.19761513866009545,
 'initial_capital': 10000,
 'final_value': 18725.96158079114,
 'sharpe_ratio': 1.5375535760838357,
 'sortino_ratio': 2.0642516198562553}

In [23]:
data = results_btc.get('signals_df')
data[data.index >= '2024-09-10'].head(15)

,asset,usd_trend_signal,btc_trend_signal,rsi_usd_signal,rsi_btc_signal,volatility_signal,ssr_signal,ssr_gate,btc_gate,btc_rsi_gate,btc_rsi_signal,followed_portfolio_signal,combined_signal,final_decision
date,,,,,,,,,,,,,,
2024-09-10,BTC,1,None,-1,None,1,-1,None,None,None,None,None,0,0
2024-09-11,BTC,1,None,-1,None,1,1,None,None,None,None,None,2,1
2024-09-12,BTC,1,None,1,None,1,1,None,None,None,None,None,4,1
2024-09-13,BTC,1,None,1,None,1,1,None,None,None,None,None,4,1
2024-09-14,BTC,1,None,1,None,1,1,None,None,None,None,None,4,1
2024-09-15,BTC,1,None,-1,None,1,1,None,None,None,None,None,2,1
2024-09-16,BTC,1,None,-1,None,1,1,None,None,None,None,None,2,1
2024-09-17,BTC,1,None,-1,None,1,1,None,None,None,None,None,2,1
2024-09-18,BTC,1,None,1,None,1,1,None,None,None,None,None,4,1


In [ ]:
# Plot performance for a specific date range
fig = analyzer.plot_individual_asset_performance(
    portfolio_name='big_wins_i',
    btc_trend_portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    show_plot=False,
   # asset_filter=['SOL']
)

In [ ]:
analyzer.asset_data['ethereum'].get('classified_data')

In [13]:
def analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='ETH', forward_days=5, start_date='2020-01-01'):
    """
    Analyze the relationship between trading signals and forward returns for a specific asset.
    
    Parameters:
    -----------
    analyzer : object
        The analyzer object with portfolio data
    portfolio_name : str
        Name of the portfolio to analyze
    asset_to_analyze : str
        Asset symbol to analyze (e.g., 'ETH', 'BTC')
    forward_days : int
        Number of days for forward returns (1, 5, or 7)
    start_date : str
        Start date for the analysis in 'YYYY-MM-DD' format
        
    Returns:
    --------
    fig : matplotlib.figure.Figure
        The figure containing the plots
    """
    # Get the signals dataframe
    results_df = analyzer.get_portfolio_details(portfolio_name=portfolio_name).get('backtest_results').get('signals_df')

    # Asset name mapping
    asset_name_map = {
        'ETH': 'ethereum',
        'BTC': 'bitcoin',
        'BNB': 'binancecoin',
        'XRP': 'ripple',
        'ADA': 'cardano',
        'SOL': 'solana',
        'DOGE': 'dogecoin',
        'DOT': 'polkadot',
        'AVAX': 'avalanche-2',
        'LINK': 'chainlink',
        'MATIC': 'matic-network'
    }

    # Filter signals for the chosen asset
    if results_df is not None and not results_df.empty:
        asset_data = results_df[results_df['asset'] == asset_to_analyze].copy()
        
        # Try to get price data from analyzer using the correct mapping
        try:
            asset_key = asset_name_map.get(asset_to_analyze, asset_to_analyze.lower())
            if asset_key in analyzer.asset_data and analyzer.asset_data[asset_key] is not None:
                # Get raw price data
                asset_price_data = analyzer.asset_data[asset_key].get('raw_data')
                
                if asset_price_data is not None:
                    # Calculate forward returns for the specified period
                    asset_price_data = asset_price_data.copy()
                    asset_price_data['daily_return'] = asset_price_data['close'].pct_change() * 100  # Convert to percentage
                    
                    # Calculate forward returns (1-day, 5-day, 7-day)
                    for days in [1, 3, 5, 7, 14]:
                        asset_price_data[f'forward_{days}d_return'] = asset_price_data['close'].shift(-days) / asset_price_data['close'] - 1
                        asset_price_data[f'forward_{days}d_return'] = asset_price_data[f'forward_{days}d_return'] * 100  # Convert to percentage
                    
                    # Merge signals with returns
                    asset_data = asset_data.reset_index()
                    asset_data['date'] = pd.to_datetime(asset_data['date'])
                    asset_price_data['date'] = pd.to_datetime(asset_price_data['date'])
                    
                    merged_data = asset_data.merge(
                        asset_price_data[['date', 'daily_return'] + [f'forward_{days}d_return' for days in [1, 3, 5, 7, 14]]], 
                        on='date', 
                        how='left'
                    ).set_index('date')
                else:
                    asset_data['daily_return'] = 0
                    for days in [1, 3, 5, 7, 14]:
                        asset_data[f'forward_{days}d_return'] = 0
                    merged_data = asset_data
            else:
                asset_data['daily_return'] = 0
                for days in [1, 3, 5, 7, 14]:
                    asset_data[f'forward_{days}d_return'] = 0
                merged_data = asset_data
        except Exception as e:
            asset_data['daily_return'] = 0
            for days in [1, 3, 5, 7, 14]:
                asset_data[f'forward_{days}d_return'] = 0
            merged_data = asset_data
        
        # Create violin plots for each signal type vs returns
        import matplotlib.pyplot as plt
        import seaborn as sns

        merged_data = merged_data[merged_data.index >= start_date]
        
        # Define signal columns to plot
        signal_columns = [
            'usd_trend_signal', 'btc_trend_signal', 'rsi_usd_signal',
            'rsi_btc_signal', 'volatility_signal', 'combined_signal', 'final_decision'
        ]
        
        # Filter to only include columns that exist in the data
        signal_columns = [col for col in signal_columns if col in merged_data.columns]
        
        # Set up the figure with a clean style
        plt.style.use('seaborn-v0_8-whitegrid')
        
        # Create a single figure with subplots
        fig, axes = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)
        axes = axes.flatten()
        
        # Select the return column based on forward_days
        return_column = f'forward_{forward_days}d_return'
        
        # Create violin plots
        for i, signal in enumerate(signal_columns):
            if i < len(axes):
                # Filter out None and NaN values
                plot_data = merged_data[merged_data[signal].notna() & merged_data[return_column].notna()].copy()
                
                # Convert signal to numeric if possible
                try:
                    plot_data[signal] = pd.to_numeric(plot_data[signal])
                except:
                    pass
                
                # Create violin plot
                sns.violinplot(
                    data=plot_data, 
                    x=signal, 
                    y=return_column,
                    ax=axes[i],
                    inner='quartile',  # Show quartiles inside violin
                    palette='viridis'
                )
                
                # Add individual points for better visualization
                sns.stripplot(
                    data=plot_data, 
                    x=signal, 
                    y=return_column,
                    ax=axes[i],
                    color='black',
                    alpha=0.3,
                    size=2,
                    jitter=True
                )
                
                # Improve aesthetics
                axes[i].set_title(f'{signal} vs {forward_days}-Day Forward Return for {asset_to_analyze}', fontsize=12, fontweight='bold')
                axes[i].axhline(y=0, color='r', linestyle='--', alpha=0.5)
                axes[i].set_xlabel(signal, fontsize=10)
                axes[i].set_ylabel(f'{forward_days}-Day Forward Return (%)', fontsize=10)
                
                # Rotate x-axis labels if there are many unique values
                if len(plot_data[signal].unique()) > 5:
                    axes[i].tick_params(axis='x', rotation=45)
        
        # Hide any unused subplots
        for j in range(len(signal_columns), len(axes)):
            axes[j].set_visible(False)
        
        # Add a title to the entire figure
        fig.suptitle(f'Signal vs {forward_days}-Day Forward Return Analysis for {asset_to_analyze}', fontsize=16, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        return fig
    
    return None

# Example usage:
# fig = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='ETH', forward_days=5)
# plt.show()


In [ ]:
fig_link = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='LINK', forward_days=3, start_date='2020-01-01')
fig_sol = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='SOL', forward_days=3, start_date='2020-01-01')
fig_eth = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='ETH', forward_days=3, start_date='2020-01-01')
fig_ada = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='ADA', forward_days=3, start_date='2020-01-01')
fig_matic = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='MATIC', forward_days=3, start_date='2020-01-01')
fig_avax = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='AVAX', forward_days=3, start_date='2020-01-01')
fig_bnb = analyze_signal_vs_returns(analyzer, portfolio_name='big_wins_i', asset_to_analyze='BNB', forward_days=3, start_date='2020-01-01')



In [ ]:
def get_signals_for_period(start_date, end_date, assets='all'):
    results_df = analyzer.get_portfolio_details(portfolio_name='altcoins_trend_following_btc').get('backtest_results').get('signals_df')
    
    # Filter by date range
    signals = results_df[(results_df.index >= start_date) & (results_df.index <= end_date)]
    
    # Filter by assets if specified
    if assets != 'all':
        if isinstance(assets, str):
            assets = [assets]
        signals = signals[signals['asset'].isin(assets)]
        
    return signals

# Example usage:
start_date = '2024-7-31'
end_date = '2025-3-25' 

assets = ['ETH']  # or 'all' for all assets
#assets = 'all'
signals = get_signals_for_period(start_date, end_date, assets)
signals[(signals['followed_portfolio_signal'] == 1) & (signals['final_decision'] == 0)]

In [ ]:
assets_held = analyzer.get_portfolio_details(portfolio_name='btc_gated_rsi_vol_momentum').get('backtest_results').get('results_df')
assets_held

import matplotlib.pyplot as plt

# Get the results dataframe

# Create the plot
plt.figure(figsize=(14, 6))

# Plot the assets held line
plt.plot(assets_held.index, assets_held['Assets_Held'], 'b-')

# Add vertical lines for each month
for date in assets_held.index[assets_held.index.is_month_start]:
    plt.axvline(x=date, color='gray', alpha=0.3, linestyle='--')

plt.title('Assets Held Over Time')
plt.ylabel('Number of Assets')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
test_criteria = [{'Short Term (USD)': ['Strong Bull'],
                  'Short Term (BTC)': ['Strong Bull'],
                  'Overall (BTC)': ['Strong Bull']},
                  
                 {'Short Term (BTC)': ['Strong Bull'],
                  'Short Term (USD)': ['Weak Bull', 'Strong Bull']}]

In [ ]:
btc_data = analyzer.asset_data['maker'].get('classified_data')[['date', 'Short Term (USD)', 'Overall (USD)', 'RSI_Signal_close']]
btc_data[btc_data['date'] == data]


In [ ]:
results_df = analyzer.get_portfolio_details(portfolio_name='altcoins_trend_following_btc').get('backtest_results')
results_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Define the assets we want to analyze
assets = {
    'BTC': 'bitcoin',
    'ETH': 'ethereum', 
    'SOL': 'solana',
    'BNB': 'binancecoin',
    'LINK': 'chainlink',
    'AVAX': 'avalanche-2'
}

# Function to plot the relationship between trend signals and forward returns
def plot_trend_signal_returns(asset_name, forward_days=7):
    # Get the data for the selected asset
    asset_data = analyzer.asset_data[asset_name].get('classified_data').copy()
    
    # Calculate forward returns
    asset_data[f'forward_returns_{forward_days}d'] = asset_data['returns_usd'].rolling(window=forward_days).sum().shift(-forward_days)
    
    # Drop NaN values
    asset_data = asset_data.dropna(subset=[f'forward_returns_{forward_days}d'])
    
    # Define the trend signals to analyze
    trend_signals = ['Short Term (USD)', 'Medium Term (USD)', 'Overall (USD)', 
                     'Short Term (BTC)', 'Medium Term (BTC)', 'Overall (BTC)']
    
    # Create a figure with subplots for each trend signal
    fig = plt.figure(figsize=(20, 30))
    gs = GridSpec(len(trend_signals), 1, figure=fig)
    
    # Plot each trend signal
    for i, signal in enumerate(trend_signals):
        if signal in asset_data.columns:
            ax = fig.add_subplot(gs[i, 0])
            
            # Create violin plot
            sns.violinplot(x=signal, y=f'forward_returns_{forward_days}d', data=asset_data, 
                          palette='coolwarm', inner='quartile', ax=ax, alpha=0.7)
            
            # Add swarm plot with reduced size and transparency
            sns.swarmplot(x=signal, y=f'forward_returns_{forward_days}d', data=asset_data, 
                         color='black', size=2, alpha=0.1, ax=ax)
            
            # Add a horizontal line at y=0
            ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
            
            # Set title and labels
            ax.set_title(f'{signal} vs {forward_days}-Day Forward Returns for {asset_name}', fontsize=14)
            ax.set_ylabel(f'{forward_days}-Day Forward Returns', fontsize=12)
            ax.set_xlabel(signal, fontsize=12)
            
            # Rotate x-axis labels for better readability
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    return fig




In [ ]:
plot_trend_signal_returns('solana', forward_days=7)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Get returns data for all assets
assets = {
    'BTC': 'bitcoin',
    'ETH': 'ethereum', 
    'SOL': 'solana',
    'BNB': 'binancecoin',
    'LINK': 'chainlink',
    'AVAX': 'avalanche-2'
}

# Create dataframe with returns for all assets
returns_df = pd.DataFrame()
for symbol, name in assets.items():
    data = analyzer.asset_data.get(name).get('classified_data')[['date','returns_usd']]
    data.set_index('date', inplace=True)
    returns_df[symbol] = data['returns_usd']

returns_df.dropna(inplace=True)

# Calculate rolling correlations against BTC
window_size = 14  # Using a 30-day window for better visualization
corr_df = pd.DataFrame(index=returns_df.index)

for col in returns_df.columns:
    if col != 'BTC':
        # Calculate rolling correlation between this asset and BTC
        corr = returns_df['BTC'].rolling(window=window_size).corr(returns_df[col])
        corr_df[col] = corr

# Drop NaN values (first window_size-1 rows will be NaN)
corr_df.dropna(inplace=True)

# Create the heatmap using plotly
asset_list = [col for col in corr_df.columns]
dates = corr_df.index

# Create the heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_df.values.T,  # Transpose to get assets on y-axis
    x=dates,
    y=asset_list,
    colorscale='RdBu_r',  # Red-Blue diverging colorscale
    zmid=0,  # Center the colorscale at 0
    zmin=-1,
    zmax=1,
    colorbar=dict(
        title=f"{window_size}-day Rolling Correlation with BTC",
        titleside="right"
    )
))

# Update layout
fig.update_layout(
    title=f"Rolling {window_size}-day Correlation of Assets with BTC",
    xaxis_title="Date",
    yaxis_title="Asset",
    height=600,
    width=1000,
    yaxis=dict(
        tickmode='array',
        tickvals=asset_list
    ),
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    annotations=[
        dict(
            x=0.5,
            y=-0.15,
            xref="paper",
            yref="paper",
            text="Blue = negative correlation, Red = positive correlation with BTC",
            showarrow=False,
            font=dict(size=12)
        )
    ]
)

# Show the figure
fig.show()
